# Reporte de predicciones — EfficientNetV2L

Genera `src/data/reporte_efficientnetv2l.csv` con el mismo formato que `src/data/reporte_resnet.csv`:
una fila por especie con `species, precision, recall, f1-score, support`, calculado con
`sklearn.metrics.classification_report` sobre las predicciones del modelo **EfficientNetV2L**
en el conjunto de test (`src/data/images_test/images_spectograms`).

Insumos:
- Pesos: `notebooks/models/weights_EfficientNetV2L.weights.h5`
- Label encoder: `notebooks/models/label_encoder_EfficientNetV2L.pkl`
- Imágenes de test: `src/data/images_test/images_spectograms` (una carpeta por especie)

Reproducibility note: ModelTrainer.create_model() keeps the classification-head Dropout layer active during inference. This notebook reconstructs EfficientNetV2L with dropout_rate=0.3; therefore, reporte_efficientnetv2l.csv represents one archived stochastic forward-pass evaluation rather than deterministic inference. These archived predictions are the values used in the downstream comparative analyses. The separate MC Dropout uncertainty analysis aggregates 2,000 stochastic forward passes.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report

from src.model_trainer import ModelTrainer
from src.image_preprocessor import ImagePreprocessor

In [ ]:
model_name = "EfficientNetV2L"

WEIGHTS_PATH = f"./models/weights_{model_name}.weights.h5"
LABEL_ENCODER_PATH = f"./models/label_encoder_{model_name}.pkl"
TEST_IMAGES_DIR = "../src/data/images_test/images_spectograms"
OUTPUT_CSV = "../src/data/reporte_efficientnetv2l.csv"

BATCH_SIZE = 64

In [ ]:
with open(LABEL_ENCODER_PATH, "rb") as f:
    label_encoder = pickle.load(f)

n_classes = len(label_encoder.classes_)
print(f"Clases en el label encoder: {n_classes}")

In [ ]:
model_trainer = ModelTrainer(
    model_name=model_name,
    img_shape=(128, 256, 1),
    n_classes=n_classes,
    dropout_rate=0.3,
    label_smoothing=0.1,
    fine_tune_layers=200
)

model = model_trainer.create_model()
model.load_weights(WEIGHTS_PATH)
print(f"Pesos cargados desde {WEIGHTS_PATH}")

In [10]:
preprocessor = ImagePreprocessor(label_encoder=label_encoder)
data = preprocessor.load_data_from_directory(TEST_IMAGES_DIR)

print(f"Imágenes de test: {len(data)} | Especies: {data['label'].nunique()}")
data.head()

Imágenes de test: 32279 | Especies: 667


,label,image_path
0,Acropternis orthonyx,../src/data/images_test/images_spectograms/Acr...
1,Acropternis orthonyx,../src/data/images_test/images_spectograms/Acr...
2,Acropternis orthonyx,../src/data/images_test/images_spectograms/Acr...
3,Acropternis orthonyx,../src/data/images_test/images_spectograms/Acr...
4,Acropternis orthonyx,../src/data/images_test/images_spectograms/Acr...


In [ ]:
# Dataset de inferencia: sin etiquetas, sin aumentación, sin cache en memoria.
ds = tf.data.Dataset.from_tensor_slices(data["image_path"].values)
ds = ds.map(preprocessor.read_image, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

logits = model.predict(ds, verbose=1)
pred_label = np.argmax(logits, axis=1)
true_label = label_encoder.transform(data["label"].values)

print(f"Predicciones: {len(pred_label)} | Accuracy global: {(pred_label == true_label).mean():.4f}")

In [8]:
true_label_str = label_encoder.inverse_transform(true_label)
pred_label_str = label_encoder.inverse_transform(pred_label)

reporte = classification_report(true_label_str, pred_label_str, output_dict=True, zero_division=0)

# Mismo formato que src/data/reporte_resnet.csv: solo una fila por especie.
df_reporte = (
    pd.DataFrame(reporte)
    .T
    .drop(index=["accuracy", "macro avg", "weighted avg"])
    .reset_index()
    .rename(columns={"index": "species"})
)
df_reporte["support"] = df_reporte["support"].astype(int)
df_reporte[["precision", "recall", "f1-score"]] = df_reporte[["precision", "recall", "f1-score"]].round(2)
df_reporte = df_reporte.sort_values("species").reset_index(drop=True)

df_reporte.to_csv(OUTPUT_CSV, index=False, float_format="%.2f")
print(f"Reporte guardado en {OUTPUT_CSV} ({len(df_reporte)} especies)")
print(f"\nMétricas globales:\n"
      f"  accuracy:    {reporte['accuracy']:.4f}\n"
      f"  macro avg:   precision={reporte['macro avg']['precision']:.4f} "
      f"recall={reporte['macro avg']['recall']:.4f} f1={reporte['macro avg']['f1-score']:.4f}\n"
      f"  weighted avg: precision={reporte['weighted avg']['precision']:.4f} "
      f"recall={reporte['weighted avg']['recall']:.4f} f1={reporte['weighted avg']['f1-score']:.4f}")

df_reporte.head(10)

Reporte guardado en ../src/data/reporte_efficientnetv2l.csv (667 especies)

Métricas globales:
  accuracy:    0.9448
  macro avg:   precision=0.9456 recall=0.9424 f1=0.9430
  weighted avg: precision=0.9465 recall=0.9448 f1=0.9449


,species,precision,recall,f1-score,support
0,Acropternis orthonyx,1.00,0.99,0.99,75
1,Amblycercus holosericeus,0.95,0.78,0.86,27
2,Ammodramus aurifrons,0.97,0.97,0.97,58
3,Ammodramus humeralis,0.91,0.93,0.92,74
4,Ammodramus savannarum,0.97,0.95,0.96,75
5,Anabacerthia striaticollis,0.96,0.93,0.94,27
6,Anabacerthia variegaticeps,0.93,0.98,0.96,44
7,Anairetes parulus,1.00,0.89,0.94,36
8,Andigena nigrirostris,0.91,0.97,0.94,63
9,Anisognathus igniventris,0.97,1.00,0.99,34
